# Build Manifest

In [2]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_0_v2.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")

study folders: 0it [00:00, ?it/s]

found 215,114 study dirs


studies processed:   0%|          | 0/215114 [00:00<?, ?it/s]


local gzip complete  •  12,109,030 PNGs  •  30.5 min
uploading …
✓ manifest at s3://echodata25/results/echo-images/all_unmasked_png_paths_0_v2.clean.txt.gz


In [3]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study-1/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_1_v2.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")

study folders: 0it [00:00, ?it/s]

found 26,145 study dirs


studies processed:   0%|          | 0/26145 [00:00<?, ?it/s]


local gzip complete  •  1,584,665 PNGs  •  4.0 min
uploading …
✓ manifest at s3://echodata25/results/echo-images/all_unmasked_png_paths_1_v2.clean.txt.gz


In [4]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study-2/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_2_v2.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")

study folders: 0it [00:00, ?it/s]

found 79,598 study dirs


studies processed:   0%|          | 0/79598 [00:00<?, ?it/s]


local gzip complete  •  5,059,180 PNGs  •  12.5 min
uploading …
✓ manifest at s3://echodata25/results/echo-images/all_unmasked_png_paths_2_v2.clean.txt.gz


# Combine Dataframes

In [23]:
import pandas as pd
from pathlib import Path

LOCAL_DIR = Path("class_preds_es0")   # your local folder with preds_rank*.csv

paths = sorted(LOCAL_DIR.glob("preds_rank*.csv"))
print(f"found {len(paths)} files")

prob_cols = [
    "a2c","a3c","a4c","a5c","plax","tee","exclude",
    "psax-av","psax-mv","psax-ap","psax-pm"
]
dtypes = {"quality": "float32", "salience": "float32", **{f"p_{v}": "float32" for v in prob_cols}}

dfs = [pd.read_csv(p, dtype=dtypes) for p in paths]

es0 = pd.concat(dfs, ignore_index=True)
print(es0.shape)

found 8 files
(5835478, 16)


In [24]:
import pandas as pd
from pathlib import Path

LOCAL_DIR = Path("class_preds_es1")   # your local folder with preds_rank*.csv

paths = sorted(LOCAL_DIR.glob("preds_rank*.csv"))
print(f"found {len(paths)} files")

prob_cols = [
    "a2c","a3c","a4c","a5c","plax","tee","exclude",
    "psax-av","psax-mv","psax-ap","psax-pm"
]
dtypes = {"quality": "float32", "salience": "float32", **{f"p_{v}": "float32" for v in prob_cols}}

dfs = [pd.read_csv(p, dtype=dtypes) for p in paths]

es1 = pd.concat(dfs, ignore_index=True)
print(es1.shape)

found 8 files
(761856, 16)


In [25]:
import pandas as pd
from pathlib import Path

LOCAL_DIR = Path("class_preds_es2")   # your local folder with preds_rank*.csv

paths = sorted(LOCAL_DIR.glob("preds_rank*.csv"))
print(f"found {len(paths)} files")

prob_cols = [
    "a2c","a3c","a4c","a5c","plax","tee","exclude",
    "psax-av","psax-mv","psax-ap","psax-pm"
]
dtypes = {"quality": "float32", "salience": "float32", **{f"p_{v}": "float32" for v in prob_cols}}

dfs = [pd.read_csv(p, dtype=dtypes) for p in paths]

es2 = pd.concat(dfs, ignore_index=True)
print(es2.shape)

found 8 files
(3671333, 16)


In [26]:
all_es = pd.concat([es0, es1, es2], ignore_index=True)

In [27]:
all_es.shape

(10268667, 16)

In [28]:
all_es.head()

,png_uri,mp4_uri,pred_view,quality,salience,p_a2c,p_a3c,p_a4c,p_a5c,p_plax,p_tee,p_exclude,p_psax-av,p_psax-mv,p_psax-ap,p_psax-pm
0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/unmasked/png/1.2.276.0.7230010.3.1.4.1714578744.1.1703116777.8010780.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/1.2.276.0.7230010.3.1.4.1714578744.1.1703116777.8010780.mp4,a4c,0.052855,0.712146,0.000517,0.000196,0.994141,0.004242,0.000000,0.000862,0.000250,0.0,0.0,0.0,0.0
1,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/unmasked/png/1.2.276.0.7230010.3.1.4.859333938.1.1703116847.8058136.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/1.2.276.0.7230010.3.1.4.859333938.1.1703116847.8058136.mp4,a2c,0.063266,0.711851,0.989746,0.000444,0.008499,0.000007,0.000029,0.000107,0.001095,0.0,0.0,0.0,0.0
2,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703120244.10466766/unmasked/png/1.2.276.0.7230010.3.1.4.845494328.1.1703120863.13455680.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703120244.10466766/1.2.276.0.7230010.3.1.4.845494328.1.1703120863.13455680.mp4,exclude,0.032048,0.709321,0.000124,0.000000,0.000104,0.000035,0.000023,0.000086,0.999512,0.0,0.0,0.0,0.0
3,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703120150.10464915.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/1.2.276.0.7230010.3.1.4.1714512485.1.1703120150.10464915.mp4,a2c,0.064069,0.537287,0.739746,0.001215,0.042572,0.000020,0.003452,0.020630,0.192261,0.0,0.0,0.0,0.0
4,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/unmasked/png/1.2.276.0.7230010.3.1.4.811753780.1.1703119932.8309776.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/1.2.276.0.7230010.3.1.4.811753780.1.1703119932.8309776.mp4,a4c,0.067666,0.680456,0.037964,0.000567,0.942871,0.013634,0.000050,0.000624,0.004345,0.0,0.0,0.0,0.0


In [29]:
import pandas as pd

# Allow pandas to display the full content of columns without truncation
pd.set_option('display.max_colwidth', None)

# Select and display the first 5 rows of the desired columns
all_es[['png_uri', 'mp4_uri']].head()

,png_uri,mp4_uri
0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/unmasked/png/1.2.276.0.7230010.3.1.4.1714578744.1.1703116777.8010780.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/1.2.276.0.7230010.3.1.4.1714578744.1.1703116777.8010780.mp4
1,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/unmasked/png/1.2.276.0.7230010.3.1.4.859333938.1.1703116847.8058136.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/1.2.276.0.7230010.3.1.4.859333938.1.1703116847.8058136.mp4
2,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703120244.10466766/unmasked/png/1.2.276.0.7230010.3.1.4.845494328.1.1703120863.13455680.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703120244.10466766/1.2.276.0.7230010.3.1.4.845494328.1.1703120863.13455680.mp4
3,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703120150.10464915.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/1.2.276.0.7230010.3.1.4.1714512485.1.1703120150.10464915.mp4
4,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/unmasked/png/1.2.276.0.7230010.3.1.4.811753780.1.1703119932.8309776.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/1.2.276.0.7230010.3.1.4.811753780.1.1703119932.8309776.mp4


# Check MP4 Paths

In [8]:
import pandas as pd
import boto3
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

S3 = boto3.client("s3")

def list_all_keys(bucket: str, prefix: str) -> list[str]:
    """List all object keys under one prefix."""
    paginator = S3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        keys.extend(obj["Key"] for obj in page.get("Contents", []))
    return keys

def _strip_bucket(uri: str, bucket: str) -> str:
    # s3://bucket/KEY... -> KEY...
    return uri.split(f"s3://{bucket}/", 1)[1]

def _check_prefix_group(args):
    """Worker: for one prefix, mark which rows exist."""
    prefix_uri, sub_df, bucket, col = args

    expected_keys = sub_df[col].map(lambda u: _strip_bucket(u, bucket))
    prefix_key   = _strip_bucket(prefix_uri, bucket)

    actual = set(list_all_keys(bucket, prefix_key))
    return expected_keys.isin(actual)

def mark_exists_by_listing(df: pd.DataFrame,
                           col: str = "mp4_uri",
                           workers: int = 64,
                           limit: int | None = None,
                           subset_only: bool = True) -> pd.Series:
    """
    Check S3 existence for the first `limit` rows (or all if None).
    Returns a boolean Series aligned to the checked subset (subset_only=True),
    or to the full df (subset_only=False, unchecked rows -> False).
    """
    dfc = df.iloc[:limit].copy() if limit is not None else df.copy()
    if dfc.empty:
        return pd.Series(dtype=bool)

    # assume single bucket
    bucket = dfc[col].iloc[0].split("/", 3)[2]

    # parent prefix of each file
    dfc["_prefix"] = dfc[col].map(lambda u: u.rsplit("/", 1)[0] + "/")

    tasks = [(p, sub, bucket, col) for p, sub in dfc.groupby("_prefix")]

    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool, \
         tqdm(total=len(tasks), desc="prefixes", unit="pfx") as bar:
        futs = {pool.submit(_check_prefix_group, t): t for t in tasks}
        for fut in as_completed(futs):
            results.append(fut.result())
            bar.update(1)

    mask_subset = pd.concat(results).sort_index() if results else pd.Series(dtype=bool)

    if subset_only:
        return mask_subset

    # expand to full df
    mask_full = pd.Series(False, index=df.index)
    mask_full.loc[mask_subset.index] = mask_subset
    return mask_full

# -------- usage --------
mask = mark_exists_by_listing(all_es, limit=1000, subset_only=True)
missing = all_es.iloc[:len(mask)].loc[~mask, "mp4_uri"]
print(f"Checked {len(mask)} rows. Missing: {missing.size}")


prefixes:   0%|          | 0/132 [00:00<?, ?pfx/s]

Checked 1000 rows. Missing: 1000


In [9]:
# import boto3, time
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from tqdm.auto import tqdm

# # --- Configuration ---
# BUCKET            = "echodata25"
# ROOT_PREFIX       = "results/echo-study/"
# DCM_SERIES_PREFIX = "1.2.276"
# THREADS           = 64
# STUDY_PAGE        = 1_000
# SCAN_LIMIT        = None # Set to a number (e.g., 1000) to test on a subset

# s3 = boto3.client("s3")

# # ────────────────────────── 1. collect study prefixes ───────────────────────
# study_pref = []
# # Using a paginator is robust for listing all study folders
# for page in s3.get_paginator("list_objects_v2").paginate(
#         Bucket=BUCKET, Prefix=ROOT_PREFIX, Delimiter="/",
#         PaginationConfig={'PageSize': STUDY_PAGE}):
#     study_pref += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

# print(f"Found {len(study_pref):,} total studies.")

# if SCAN_LIMIT:
#     print(f"⚠️  Limiting scan to the first {SCAN_LIMIT:,} studies.")
#     study_pref = study_pref[:SCAN_LIMIT]

# # ──── 2. Find studies with >1 DICOM series directory in parallel ─────────
# def check_for_multiple_dicom_dirs(prefix):
#     """
#     Checks if a study has more than one subdirectory starting with '1.2.276'.
#     Returns the prefix if it matches, otherwise None.
#     """
#     dicom_dir_count = 0
#     paginator = s3.get_paginator("list_objects_v2")
#     try:
#         # List immediate subdirectories of the study
#         for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix, Delimiter="/"):
#             for cp in page.get("CommonPrefixes", []):
#                 # Extract subdirectory name, e.g., "1.2.276.xyz"
#                 sub_dir_name = cp["Prefix"].strip("/").split("/")[-1]
#                 if sub_dir_name.startswith(DCM_SERIES_PREFIX):
#                     dicom_dir_count += 1
                
#                 # Optimization: stop scanning as soon as the condition is met
#                 if dicom_dir_count > 1:
#                     raise StopIteration
#     except StopIteration:
#         return prefix # Condition met, return the study prefix
    
#     return None # Condition not met after checking all subdirectories

# matching_studies = []
# t0 = time.time()
# with ThreadPoolExecutor(max_workers=THREADS) as pool:
#     futures = {pool.submit(check_for_multiple_dicom_dirs, p) for p in study_pref}
#     for fut in tqdm(as_completed(futures), total=len(futures), desc="Scanning studies"):
#         result = fut.result()
#         if result:
#             matching_studies.append(result)

# elapsed = time.time() - t0

# # ────────────────────────── 3. Print the results ────────────────────────────
# print("\n" + "="*80)
# print(f"Scan complete in {elapsed:.1f} seconds.")
# if matching_studies:
#     print(f"✅ Found {len(matching_studies)} studies with more than one '{DCM_SERIES_PREFIX}...' subdirectory:")
#     for prefix in sorted(matching_studies):
#         print(f"s3://{BUCKET}/{prefix}")
# else:
#     print(f"ℹ️ No studies found matching the criteria.")
# print("="*80)

In [10]:
# import re
# import os
# import pandas as pd
# import boto3
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from tqdm.auto import tqdm

# # -------------------- config --------------------
# URI_COL   = "png_uri"          # column in your DF
# OUT_COL   = "mp4_uri_corrected"          # new column to fill
# WORKERS   = 64
# BUCKET_RE = re.compile(r"^s3://([^/]+)/")
# S3 = boto3.client("s3")

# # -------------------- helpers --------------------
# def s3_bucket_key(uri: str) -> tuple[str, str]:
#     m = BUCKET_RE.match(uri)
#     if not m:
#         raise ValueError(f"bad s3 uri: {uri}")
#     b = m.group(1)
#     return b, uri[len(f"s3://{b}/"):]

# def study_root_from_png(uri: str) -> str:
#     # everything up to ".../unmasked/"
#     return uri.split("/unmasked/", 1)[0] + "/"

# def basename_no_ext(uri: str) -> str:
#     return os.path.basename(uri).rsplit(".", 1)[0]

# def list_all_keys(bucket: str, prefix: str) -> list[str]:
#     paginator = S3.get_paginator("list_objects_v2")
#     keys = []
#     for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
#         keys.extend(obj["Key"] for obj in page.get("Contents", []) if "Contents" in page)
#     return keys

# def build_mapping_for_study(study_prefix: str, bucket: str) -> dict[str, str]:
#     # list once, keep only wanted mp4s (exclude unmasked/png, mask_visualization)
#     prefix_key = study_prefix.replace(f"s3://{bucket}/", "", 1)
#     all_keys = list_all_keys(bucket, prefix_key)
#     mp4_keys = [
#         k for k in all_keys
#         if k.endswith(".mp4")
#         and "/unmasked/" not in k
#         and not k.endswith("mask_visualization.mp4")
#     ]
#     return {os.path.basename(k)[:-4]: f"s3://{bucket}/{k}" for k in mp4_keys}

# # -------------------- main --------------------
# def map_png_to_mp4(df: pd.DataFrame,
#                    uri_col: str = URI_COL,
#                    out_col: str = OUT_COL,
#                    workers: int = WORKERS) -> pd.Series:
#     if df.empty:
#         return pd.Series(dtype=object, index=df.index)

#     bucket = BUCKET_RE.match(df[uri_col].iloc[0]).group(1)

#     # derive study prefix and basename
#     tmp = pd.DataFrame({
#         "uri": df[uri_col].values,
#         "study_prefix": df[uri_col].map(study_root_from_png),
#         "basename": df[uri_col].map(basename_no_ext),
#     }, index=df.index)

#     # group by study to minimize S3 calls
#     groups = list(tmp.groupby("study_prefix"))

#     # fetch mappings in parallel
#     mappings = {}
#     with ThreadPoolExecutor(max_workers=workers) as pool, \
#          tqdm(total=len(groups), desc="list S3", unit="study") as pbar:
#         futs = {
#             pool.submit(build_mapping_for_study, study, bucket): study
#             for study, _ in groups
#         }
#         for fut in as_completed(futs):
#             study = futs[fut]
#             mappings[study] = fut.result()
#             pbar.update(1)

#     # resolve for each row
#     def resolve(row):
#         return mappings.get(row["study_prefix"], {}).get(row["basename"], None)

#     return tmp.apply(resolve, axis=1)

# # -------------------- usage --------------------
# # all_es: DataFrame with a 'png_uri' column
# all_es["mp4_uri_mapped"] = map_png_to_mp4(all_es)
# missing = all_es[all_es["mp4_uri_mapped"].isna()][URI_COL]
# print(f"mapped: {all_es['mp4_uri_mapped'].notna().sum():,} | missing: {missing.size:,}")


# Fix MP4 Paths

In [30]:
import re
import os
import pandas as pd
import boto3
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# -------------------- config --------------------
URI_COL   = "png_uri"
OUT_COL   = "mp4_uri_corrected"
WORKERS   = 64
BUCKET_RE = re.compile(r"^s3://([^/]+)/")
S3 = boto3.client("s3")

# -------------------- helpers (unchanged) --------------------
def s3_bucket_key(uri: str) -> tuple[str, str]:
    m = BUCKET_RE.match(uri)
    if not m:
        raise ValueError(f"bad s3 uri: {uri}")
    b = m.group(1)
    return b, uri[len(f"s3://{b}/"):]

def study_root_from_png(uri: str) -> str:
    return uri.split("/unmasked/", 1)[0] + "/"

def basename_no_ext(uri: str) -> str:
    return os.path.basename(uri).rsplit(".", 1)[0]

def list_all_keys(bucket: str, prefix: str) -> list[str]:
    paginator = S3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if "Contents" in page:
             keys.extend(obj["Key"] for obj in page.get("Contents", []))
    return keys

def build_mapping_for_study(study_prefix: str, bucket: str) -> dict[str, str]:
    prefix_key = study_prefix.replace(f"s3://{bucket}/", "", 1)
    all_keys = list_all_keys(bucket, prefix_key)
    mp4_keys = [
        k for k in all_keys
        if k.endswith(".mp4")
        and "/unmasked/" not in k
        and not k.endswith("mask_visualization.mp4")
    ]
    return {os.path.basename(k)[:-4]: f"s3://{bucket}/{k}" for k in mp4_keys}

# -------------------- main (modified) --------------------
def map_png_to_mp4(df: pd.DataFrame,
                   uri_col: str = URI_COL,
                   out_col: str = OUT_COL,
                   workers: int = WORKERS,
                   limit: int = None,
                   view: str = None) -> pd.Series:
    if df.empty:
        return pd.Series(dtype=object, index=df.index)

    # --- Create a subset of the DataFrame to process based on filters ---
    df_to_process = df.copy()
    if view:
        if "pred_view" in df_to_process.columns:
            df_to_process = df_to_process[df_to_process["pred_view"] == view]
        else:
            print(f"Warning: 'pred_view' column not found. Cannot filter by view '{view}'.")
    if limit:
        df_to_process = df_to_process.head(limit)

    if df_to_process.empty:
        print("No rows match the specified filters.")
        return pd.Series(dtype=object, index=df.index)
    
    print(f"Processing {len(df_to_process):,} filtered rows...")

    # The rest of the function operates on the filtered `df_to_process`
    bucket = BUCKET_RE.match(df_to_process[uri_col].iloc[0]).group(1)

    tmp = pd.DataFrame({
        "uri": df_to_process[uri_col].values,
        "study_prefix": df_to_process[uri_col].map(study_root_from_png),
        "basename": df_to_process[uri_col].map(basename_no_ext),
    }, index=df_to_process.index)

    groups = list(tmp.groupby("study_prefix"))
    
    mappings = {}
    with ThreadPoolExecutor(max_workers=workers) as pool, \
         tqdm(total=len(groups), desc="list S3", unit="study") as pbar:
        futs = {
            pool.submit(build_mapping_for_study, study, bucket): study
            for study, _ in groups
        }
        for fut in as_completed(futs):
            study = futs[fut]
            mappings[study] = fut.result()
            pbar.update(1)

    def resolve(row):
        return mappings.get(row["study_prefix"], {}).get(row["basename"], None)

    # Apply the mapping to the filtered subset
    processed_results = tmp.apply(resolve, axis=1)

    # Reindex to match the original DataFrame, filling non-processed rows with NaN
    return processed_results.reindex(df.index)

# -------------------- usage --------------------
# Example: Correct only the first 1000 rows that are predicted as 'a4c'
a4c_limit = None
all_es["mp4_uri_corrected"] = map_png_to_mp4(
    all_es,
    limit=a4c_limit,
    view=None
)

# Example: Correct all 'a4c' rows (no limit)
# all_es["mp4_uri_corrected_a4c_all"] = map_png_to_mp4(all_es, view='a4c')

# Example: Correct all rows (original behavior)
# all_es["mp4_uri_corrected_all"] = map_png_to_mp4(all_es)

mapped_count = all_es['mp4_uri_corrected'].notna().sum()
print(f"Mapped {mapped_count:,} URIs for view 'a4c' with a limit of {a4c_limit}.")

Processing 10,268,667 filtered rows...


list S3:   0%|          | 0/239064 [00:00<?, ?study/s]

Mapped 10,268,667 URIs for view 'a4c' with a limit of None.


In [31]:
# Make sure you have the necessary library:
# pip install pyarrow

output_filename = 'all_es_corrected_p2.parquet'

# Save the DataFrame to a Parquet file
all_es.to_parquet(output_filename, index=False)

print(f"✅ Successfully saved DataFrame to '{output_filename}'")

✅ Successfully saved DataFrame to 'all_es_corrected_p2.parquet'


# Read File

In [32]:
# # Read the Parquet file back into a new DataFrame
output_filename = 'all_es_corrected.parquet'
all_es1 = pd.read_parquet(output_filename)

print("✅ Successfully reloaded DataFrame from Parquet file.")
print(f"Shape of reloaded DataFrame: {all_es1.shape}")
all_es1.head()

✅ Successfully reloaded DataFrame from Parquet file.
Shape of reloaded DataFrame: (7842714, 17)


,png_uri,mp4_uri,pred_view,quality,salience,p_a2c,p_a3c,p_a4c,p_a5c,p_plax,p_tee,p_exclude,p_psax-av,p_psax-mv,p_psax-ap,p_psax-pm,mp4_uri_corrected
0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.811753780.1.1703111362.8215013.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.811753780.1.1703111362.8215013.mp4,exclude,0.042402,0.541041,0.000002,0.000120,0.000119,0.000952,0.222900,0.020416,0.754395,0.001119,0.000000,0.000137,0.000011,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111362.8215013.mp4
1,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.1714578744.1.1703111407.7964352.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.1714578744.1.1703111407.7964352.mp4,a2c,0.083963,0.576947,0.788086,0.080505,0.126953,0.000072,0.000019,0.002302,0.001903,0.000000,0.000000,0.000000,0.000000,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.1714578744.1.1703111407.7964352.mp4
2,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703111411.10353378.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.1714512485.1.1703111411.10353378.mp4,tee,0.090764,0.362434,0.011147,0.000251,0.010651,0.000403,0.005856,0.478760,0.451660,0.000301,0.000902,0.012230,0.027878,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.1714512485.1.1703111411.10353378.mp4
3,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.mp4,a4c,0.062837,0.626273,0.094849,0.000040,0.867188,0.001674,0.000007,0.019531,0.016907,0.000000,0.000000,0.000000,0.000000,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.mp4
4,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/unmasked/png/1.2.276.0.7230010.3.1.4.1714578744.1.1703111393.7964326.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.4.1714578744.1.1703111393.7964326.mp4,exclude,0.024978,0.703782,0.000076,0.000004,0.000198,0.003231,0.000531,0.001554,0.994629,0.000006,0.000000,0.000001,0.000000,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.1714578744.1.1703111393.7964326.mp4


In [33]:
# # Read the Parquet file back into a new DataFrame
output_filename = 'all_es_corrected_p2.parquet'
all_es2 = pd.read_parquet(output_filename)

print("✅ Successfully reloaded DataFrame from Parquet file.")
print(f"Shape of reloaded DataFrame: {all_es2.shape}")
all_es2.head()

✅ Successfully reloaded DataFrame from Parquet file.
Shape of reloaded DataFrame: (10268667, 17)


,png_uri,mp4_uri,pred_view,quality,salience,p_a2c,p_a3c,p_a4c,p_a5c,p_plax,p_tee,p_exclude,p_psax-av,p_psax-mv,p_psax-ap,p_psax-pm,mp4_uri_corrected
0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/unmasked/png/1.2.276.0.7230010.3.1.4.1714578744.1.1703116777.8010780.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/1.2.276.0.7230010.3.1.4.1714578744.1.1703116777.8010780.mp4,a4c,0.052855,0.712146,0.000517,0.000196,0.994141,0.004242,0.000000,0.000862,0.000250,0.0,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/1.2.276.0.7230010.3.1.3.1714512485.1.1703116346.10397127/1.2.276.0.7230010.3.1.4.1714578744.1.1703116777.8010780.mp4
1,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/unmasked/png/1.2.276.0.7230010.3.1.4.859333938.1.1703116847.8058136.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/1.2.276.0.7230010.3.1.4.859333938.1.1703116847.8058136.mp4,a2c,0.063266,0.711851,0.989746,0.000444,0.008499,0.000007,0.000029,0.000107,0.001095,0.0,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703116346.10397126/1.2.276.0.7230010.3.1.3.1714512485.1.1703116346.10397127/1.2.276.0.7230010.3.1.4.859333938.1.1703116847.8058136.mp4
2,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703120244.10466766/unmasked/png/1.2.276.0.7230010.3.1.4.845494328.1.1703120863.13455680.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703120244.10466766/1.2.276.0.7230010.3.1.4.845494328.1.1703120863.13455680.mp4,exclude,0.032048,0.709321,0.000124,0.000000,0.000104,0.000035,0.000023,0.000086,0.999512,0.0,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703120244.10466766/1.2.276.0.7230010.3.1.3.1714512485.1.1703120244.10466767/1.2.276.0.7230010.3.1.4.845494328.1.1703120863.13455680.mp4
3,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/unmasked/png/1.2.276.0.7230010.3.1.4.1714512485.1.1703120150.10464915.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/1.2.276.0.7230010.3.1.4.1714512485.1.1703120150.10464915.mp4,a2c,0.064069,0.537287,0.739746,0.001215,0.042572,0.000020,0.003452,0.020630,0.192261,0.0,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/1.2.276.0.7230010.3.1.3.1714512485.1.1703119343.10450507/1.2.276.0.7230010.3.1.4.1714512485.1.1703120150.10464915.mp4
4,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/unmasked/png/1.2.276.0.7230010.3.1.4.811753780.1.1703119932.8309776.png,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/1.2.276.0.7230010.3.1.4.811753780.1.1703119932.8309776.mp4,a4c,0.067666,0.680456,0.037964,0.000567,0.942871,0.013634,0.000050,0.000624,0.004345,0.0,0.0,0.0,0.0,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119343.10450506/1.2.276.0.7230010.3.1.3.1714512485.1.1703119343.10450507/1.2.276.0.7230010.3.1.4.811753780.1.1703119932.8309776.mp4


In [34]:
import pandas as pd

# Assuming all_es1 and all_es2 are already loaded as in your image

# Combine the two DataFrames
# ignore_index=True will create a new continuous index for the combined DataFrame
combined_df = pd.concat([all_es1, all_es2], ignore_index=True)

# Define the new output filename
output_filename_combined = 'all_es_combined.parquet'

# Save the combined DataFrame to a new Parquet file
# index=False prevents pandas from writing the DataFrame index as a column
combined_df.to_parquet(output_filename_combined, index=False)

# --- Verification (Optional but recommended) ---
print("✅ Successfully combined DataFrames.")
print(f"Shape of original DataFrame 1: {all_es1.shape}")
print(f"Shape of original DataFrame 2: {all_es2.shape}")
print(f"Shape of combined DataFrame: {combined_df.shape}")
print(f"\n✅ Successfully saved combined DataFrame to '{output_filename_combined}'")

✅ Successfully combined DataFrames.
Shape of original DataFrame 1: (7842714, 17)
Shape of original DataFrame 2: (10268667, 17)
Shape of combined DataFrame: (18111381, 17)

✅ Successfully saved combined DataFrame to 'all_es_combined.parquet'


# A4C

In [35]:
# Create a new DataFrame called 'a4c_df' by filtering 'all_es'
all_es = combined_df
a4c_df = all_es[all_es['pred_view'] == 'a4c'].copy()

# This regex finds 'echo-study', 'echo-study-1', or 'echo-study-2',
# and then captures the sequence of characters that follows until the next slash.
regex_pattern = r'results/echo-study(?:-[12])?/([^/]+)'

# Use .str.extract() to pull out the captured group (the Study ID)
a4c_df['DeidentifiedStudyID'] = a4c_df['png_uri'].str.extract(regex_pattern)

a4c_df = a4c_df[['DeidentifiedStudyID', 'mp4_uri_corrected']].rename(
    columns={'mp4_uri_corrected': 'URI'}
)

In [36]:
print(a4c_df.shape)
display(a4c_df.head())

(2976025, 2)


,DeidentifiedStudyID,URI
3,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.mp4
5,1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773/1.2.276.0.7230010.3.1.3.1714512485.1.1703119859.10459774/1.2.276.0.7230010.3.1.4.1714512485.1.1703120151.10464936.mp4
9,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111413.8215093.mp4
11,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111387.8215053.mp4
13,1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/1.2.276.0.7230010.3.1.3.1714512485.1.1703113062.10354035/1.2.276.0.7230010.3.1.4.1714512485.1.1703113992.10362168.mp4


# Connect to Labels

In [37]:
mvr_labels = pd.read_csv('mvr_labels.csv')

In [38]:
# Select only the necessary columns from your labels DataFrame for efficiency
labels_to_merge = mvr_labels[['DeidentifiedStudyID', 'Value']]

# Merge the two DataFrames to find the overlap and add the 'Value'
# 'how="inner"' ensures only matching DeidentifiedStudyIDs are kept
a4c_mvr_labels = pd.merge(
    a4c_df, 
    labels_to_merge, 
    on='DeidentifiedStudyID', 
    how='inner'
)

print(f"Found {len(a4c_mvr_labels)} overlapping studies.")
a4c_mvr_labels.head()

Found 466345 overlapping studies.


,DeidentifiedStudyID,URI,Value
0,1.2.276.0.7230010.3.1.2.1714512485.1.1703360128.15623239,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703360128.15623239/1.2.276.0.7230010.3.1.3.1714512485.1.1703360152.15624003/1.2.276.0.7230010.3.1.4.859333938.1.1703361188.12739794.mp4,trace
1,1.2.276.0.7230010.3.1.2.1714512485.1.1703360831.15646786,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703360831.15646786/1.2.276.0.7230010.3.1.3.1714578744.1.1703360844.12637352/1.2.276.0.7230010.3.1.4.845494328.1.1703361296.18316453.mp4,trace
2,1.2.276.0.7230010.3.1.2.1714512485.1.1703364451.15783086,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703364451.15783086/1.2.276.0.7230010.3.1.3.1714512485.1.1703364453.15783214/1.2.276.0.7230010.3.1.4.859333938.1.1703365064.12872595.mp4,mild
3,1.2.276.0.7230010.3.1.2.1714512485.1.1703364451.15783086,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703364451.15783086/1.2.276.0.7230010.3.1.3.1714512485.1.1703364453.15783214/1.2.276.0.7230010.3.1.4.1714578744.1.1703365434.12786046.mp4,mild
4,1.2.276.0.7230010.3.1.2.1714512485.1.1703364482.15784791,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703364482.15784791/1.2.276.0.7230010.3.1.3.1714512485.1.1703364484.15784917/1.2.276.0.7230010.3.1.4.811753780.1.1703365140.12915308.mp4,moderate


In [39]:
a4c_mvr_labels.to_csv('a4c_mvr_labels.csv')